<a href="https://colab.research.google.com/github/mmuputisi/Adv-Py_Data_Analysis/blob/main/LEN_Supply_Model_v6_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog
import warnings
import os
from datetime import datetime
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ========== CONFIGURATION SECTION ==========
ENABLE_DEMAND_SCENARIOS = True
ENABLE_PRICE_SENSITIVITY = False
ENABLE_PRODUCTION_SMOOTHING = False
ENABLE_SUPPLIER_SHARE_CONSTRAINTS = True
ENABLE_MINIMUM_VOLUME_CONSTRAINTS = True
ENABLE_WHAT_IF_ANALYSIS = True
ENABLE_COUNTRY_FUNDER_SPLIT_CONSTRAINTS = True

# Production smoothing parameters
SMOOTHING_PENALTY = 0.05
MAX_QUARTERLY_CHANGE = 0.50
ENABLE_SMOOTHING_CONSTRAINTS = False

# Demand scenarios to test
DEMAND_SCENARIOS = ['base', 'optimistic', 'pessimistic', 'seasonal_peak']

# Custom demand multiplier
CUSTOM_DEMAND_MULTIPLIER = 1.15

# Price sensitivity parameters
PRICE_SENSITIVITY_PRODUCT = 'S1_G1'
PRICE_SENSITIVITY_RANGE = [-0.30, -0.20, -0.10, 0, 0.10, 0.20, 0.30]

# ========== PRODUCT MAPPING ==========
PRODUCT_MAPPING = {
    'Product_A': 'LEN_vial',
    'Product_B': 'LEN_tablet'
}

# ========== COUNTRY MAPPING ==========
COUNTRY_NAMES = [
    'Eswatini', 'Kenya', 'Lesotho', 'Malawi', 'Mozambique', 'Philippines',
    'Nigeria', 'South Africa', 'Uganda', 'Ukraine', 'Zambia', 'Zimbabwe',
    'Country_13', 'Country_14', 'Country_15', 'Country_16'
]

COUNTRY_MAPPING = {f'Country_{i+1}': COUNTRY_NAMES[i] for i in range(16)}

# Save path
SAVE_PATH = os.getcwd()
os.makedirs(SAVE_PATH, exist_ok=True)
CONFIG_PATH = os.path.join(SAVE_PATH, 'optimization_config.xlsx')
print(f"\nFiles will be saved to: {SAVE_PATH}")

# ========== DEMAND DATA (CORRECTLY ALIGNED WITH COUNTRY NAMES) ==========
# LEN_vial demand - indices match COUNTRY_NAMES order (Eswatini=0, Kenya=1, etc.)
demand_data_base_product_A = {
    'Q1': [28046, 148707, 45119, 0, 116143, 53183, 0, 230784, 71213, 0, 106147, 19341, 0, 0, 0, 0],
    'Q2': [60647, 305675, 97052, 157444, 224276, 87752, 169427, 468856, 137974, 6183, 238447, 222929, 0, 0, 0, 0],
    'Q3': [0, 308429, 58655, 311607, 195574, 0, 0, 692352, 293752, 6183, 123070, 0, 0, 0, 0, 0],
    'Q4': [47003, 172114, 97052, 157444, 83436, 106366, 169427, 688708, 222539, 0, 769185, 501761, 0, 0, 0, 0]
}

# LEN_tablet demand - indices match COUNTRY_NAMES order
demand_data_base_product_B = {
    'Q1': [32245, 235074, 40712, 0, 159151, 74920, 0, 232195, 107515, 0, 98959, 22367, 0, 0, 0, 0],
    'Q2': [67637, 149592, 136830, 237247, 159151, 74920, 227363, 471722, 86012, 9365, 241624, 304080, 0, 0, 0, 0],
    'Q3': [0, 278260, 40712, 237247, 115126, 0, 0, 351958, 335986, 0, 162457, 0, 0, 0, 0, 0],
    'Q4': [0, 0, 0, 0, 0, 37460, 0, 340226, 0, 0, 412327, 232985, 0, 0, 0, 0]
}

# Original country codes for internal use
countries_original = [f'Country_{i+1}' for i in range(16)]
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

n_quarters = 4
n_countries = 16
n_combinations = 4
n_products = 2
n_vars_per_product = n_quarters * n_countries * n_combinations

# ========== HELPER FUNCTIONS ==========
def translate_country_name(country_code):
    return COUNTRY_MAPPING.get(country_code, country_code)

def translate_dataframe_countries(df):
    if isinstance(df, pd.DataFrame):
        new_columns = [translate_country_name(col) for col in df.columns]
        df.columns = new_columns
    return df

def apply_demand_scenario(demand_data_base, scenario_name):
    if scenario_name == 'base':
        return demand_data_base
    elif scenario_name == 'optimistic':
        return {q: [int(d * 1.2) for d in demand_data_base[q]] for q in demand_data_base}
    elif scenario_name == 'pessimistic':
        return {q: [int(d * 0.8) for d in demand_data_base[q]] for q in demand_data_base}
    elif scenario_name == 'seasonal_peak':
        return {q: [int(d * 1.3) if q in ['Q2', 'Q3'] else int(d * 0.9) for d in demand_data_base[q]] for q in demand_data_base}
    else:
        return demand_data_base

# ========== FUNCTION TO CREATE CONFIGURATION EXCEL ==========
def create_configuration_excel():
    with pd.ExcelWriter(CONFIG_PATH, engine='openpyxl') as writer:
        # Prices
        prices_A_df = pd.DataFrame([
            {'Supplier_Funder': 'Gilead_GF', 'Price': 21.05, 'Co_payment': 23.86},
            {'Supplier_Funder': 'Gilead_GHSD', 'Price': 42.11, 'Co_payment': 0.00},
            {'Supplier_Funder': 'Generics_GF', 'Price': 10.00, 'Co_payment': 0.00},
            {'Supplier_Funder': 'Generics_GHSD', 'Price': 10.00, 'Co_payment': 0.00}
        ])
        prices_A_df.to_excel(writer, sheet_name='LEN_vial_Prices', index=False)

        prices_B_df = pd.DataFrame([
            {'Supplier_Funder': 'Gilead_GF', 'Price': 8.95, 'Co_payment': 10.14},
            {'Supplier_Funder': 'Gilead_GHSD', 'Price': 18.89, 'Co_payment': 0.00},
            {'Supplier_Funder': 'Generics_GF', 'Price': 4.25, 'Co_payment': 0.00},
            {'Supplier_Funder': 'Generics_GHSD', 'Price': 4.25, 'Co_payment': 0.00}
        ])
        prices_B_df.to_excel(writer, sheet_name='LEN_tablet_Prices', index=False)

        # Demand sheets with correct country order
        demand_A_data = []
        for quarter in quarters:
            for i, country in enumerate(COUNTRY_NAMES):
                demand_A_data.append({
                    'Product': 'LEN_vial',
                    'Country': country,
                    'Quarter': quarter,
                    'Demand': demand_data_base_product_A[quarter][i]
                })
        demand_A_df = pd.DataFrame(demand_A_data)
        demand_A_df = demand_A_df.pivot_table(index=['Product', 'Country'], columns='Quarter', values='Demand').reset_index()
        demand_A_df.to_excel(writer, sheet_name='LEN_vial_Demand', index=False)

        demand_B_data = []
        for quarter in quarters:
            for i, country in enumerate(COUNTRY_NAMES):
                demand_B_data.append({
                    'Product': 'LEN_tablet',
                    'Country': country,
                    'Quarter': quarter,
                    'Demand': demand_data_base_product_B[quarter][i]
                })
        demand_B_df = pd.DataFrame(demand_B_data)
        demand_B_df = demand_B_df.pivot_table(index=['Product', 'Country'], columns='Quarter', values='Demand').reset_index()
        demand_B_df.to_excel(writer, sheet_name='LEN_tablet_Demand', index=False)

        # Supplier Capacities
        capacity_df = pd.DataFrame({
            'Quarter': ['Q1', 'Q2', 'Q3', 'Q4'],
            'Gilead_Capacity_LEN_vial': [600000, 600000, 600000, 600000],
            'Generics_Capacity_LEN_vial': [1500000, 1600000, 1500000, 2500000],
            'Gilead_Capacity_LEN_tablet': [600000, 600000, 600000, 600000],
            'Generics_Capacity_LEN_tablet': [1500000, 1600000, 1500000, 2500000]
        })
        capacity_df.to_excel(writer, sheet_name='Supplier_Capacities', index=False)

        # Share Constraints
        share_constraints_df = pd.DataFrame([
            {'Product': 'LEN_vial', 'Funder': 'GF', 'Supplier': 'Gilead', 'Min_Share_%': 0, 'Max_Share_%': 100},
            {'Product': 'LEN_vial', 'Funder': 'GF', 'Supplier': 'Generics', 'Min_Share_%': 0, 'Max_Share_%': 100},
            {'Product': 'LEN_vial', 'Funder': 'GHSD', 'Supplier': 'Gilead', 'Min_Share_%': 0, 'Max_Share_%': 70},
            {'Product': 'LEN_vial', 'Funder': 'GHSD', 'Supplier': 'Generics', 'Min_Share_%': 0, 'Max_Share_%': 30},
            {'Product': 'LEN_tablet', 'Funder': 'GF', 'Supplier': 'Gilead', 'Min_Share_%': 0, 'Max_Share_%': 100},
            {'Product': 'LEN_tablet', 'Funder': 'GF', 'Supplier': 'Generics', 'Min_Share_%': 0, 'Max_Share_%': 100},
            {'Product': 'LEN_tablet', 'Funder': 'GHSD', 'Supplier': 'Gilead', 'Min_Share_%': 0, 'Max_Share_%': 70},
            {'Product': 'LEN_tablet', 'Funder': 'GHSD', 'Supplier': 'Generics', 'Min_Share_%': 0, 'Max_Share_%': 30}
        ])
        share_constraints_df.to_excel(writer, sheet_name='Share_Constraints', index=False)

        # Minimum Volume Requirements
        min_volume_df = pd.DataFrame([
            {'Product': 'LEN_vial', 'Supplier': 'Gilead', 'Funder': 'GF', 'Min_Annual_Volume': 0},
            {'Product': 'LEN_vial', 'Supplier': 'Gilead', 'Funder': 'GHSD', 'Min_Annual_Volume': 0},
            {'Product': 'LEN_vial', 'Supplier': 'Generics', 'Funder': 'GF', 'Min_Annual_Volume': 0},
            {'Product': 'LEN_vial', 'Supplier': 'Generics', 'Funder': 'GHSD', 'Min_Annual_Volume': 0},
            {'Product': 'LEN_tablet', 'Supplier': 'Gilead', 'Funder': 'GF', 'Min_Annual_Volume': 0},
            {'Product': 'LEN_tablet', 'Supplier': 'Gilead', 'Funder': 'GHSD', 'Min_Annual_Volume': 0},
            {'Product': 'LEN_tablet', 'Supplier': 'Generics', 'Funder': 'GF', 'Min_Annual_Volume': 0},
            {'Product': 'LEN_tablet', 'Supplier': 'Generics', 'Funder': 'GHSD', 'Min_Annual_Volume': 0}
        ])
        min_volume_df.to_excel(writer, sheet_name='Minimum_Volumes', index=False)

        # Country Funder Split Constraints
        country_split_data = []
        for i, country in enumerate(COUNTRY_NAMES):
            # Default: no constraints (0-100%)
            min_gf, max_gf, min_ghsd, max_ghsd = 0, 100, 0, 100

            # Apply specific constraints
            if country in ['Malawi', 'Philippines', 'Ukraine']:
                max_gf = 0  # GF not allowed (only GHSD)
            if country in ['Nigeria', 'South Africa']:
                max_ghsd = 0  # GHSD not allowed (only GF)

            country_split_data.append({
                'Product': 'LEN_vial',
                'Country': country,
                'Min_GF_Share_%': min_gf,
                'Max_GF_Share_%': max_gf,
                'Min_GHSD_Share_%': min_ghsd,
                'Max_GHSD_Share_%': max_ghsd,
                'Notes': ''
            })
            country_split_data.append({
                'Product': 'LEN_tablet',
                'Country': country,
                'Min_GF_Share_%': min_gf,
                'Max_GF_Share_%': max_gf,
                'Min_GHSD_Share_%': min_ghsd,
                'Max_GHSD_Share_%': max_ghsd,
                'Notes': ''
            })

        country_split_df = pd.DataFrame(country_split_data)
        country_split_df.to_excel(writer, sheet_name='Country_Funder_Split', index=False)

        # What-If Scenarios
        whatif_scenarios_df = pd.DataFrame([
            {'Scenario_Name': 'Base_Constraints', 'Product': 'LEN_vial', 'Funder': 'GHSD',
             'Supplier': 'Generics', 'Max_Share_%': 30, 'Description': 'Current constraints'}
        ])
        whatif_scenarios_df.to_excel(writer, sheet_name='WhatIf_Scenarios', index=False)

        # Country Mapping
        country_mapping_df = pd.DataFrame({
            'Internal_Code': countries_original,
            'Country_Name': COUNTRY_NAMES,
            'Notes': ['Real country'] * 12 + ['Placeholder'] * 4
        })
        country_mapping_df.to_excel(writer, sheet_name='Country_Mapping', index=False)

        # Instructions
        instructions = pd.DataFrame({'Instruction': [
            '1. Edit values to modify configuration',
            '2. LEN_vial and LEN_tablet prices can be changed',
            '3. Supplier capacities can be modified',
            '4. Country Funder Split: Set min/max % of GF/GHSD per country (0-100%)',
            '5. Save the file and re-run the optimization'
        ]})
        instructions.to_excel(writer, sheet_name='Instructions', index=False)

    print(f"✓ Configuration template created: {CONFIG_PATH}")

# ========== FUNCTION TO LOAD CONFIGURATION ==========
def load_configuration():
    prices_A_df = pd.read_excel(CONFIG_PATH, sheet_name='LEN_vial_Prices')
    prices_B_df = pd.read_excel(CONFIG_PATH, sheet_name='LEN_tablet_Prices')

    prices_product_A = {
        'S1_G1': prices_A_df[prices_A_df['Supplier_Funder'] == 'Gilead_GF']['Price'].values[0],
        'S1_G2': prices_A_df[prices_A_df['Supplier_Funder'] == 'Gilead_GHSD']['Price'].values[0],
        'S2_G1': prices_A_df[prices_A_df['Supplier_Funder'] == 'Generics_GF']['Price'].values[0],
        'S2_G2': prices_A_df[prices_A_df['Supplier_Funder'] == 'Generics_GHSD']['Price'].values[0]
    }
    co_payment_product_A = {
        'S1_G1': prices_A_df[prices_A_df['Supplier_Funder'] == 'Gilead_GF']['Co_payment'].values[0],
        'S1_G2': prices_A_df[prices_A_df['Supplier_Funder'] == 'Gilead_GHSD']['Co_payment'].values[0],
        'S2_G1': prices_A_df[prices_A_df['Supplier_Funder'] == 'Generics_GF']['Co_payment'].values[0],
        'S2_G2': prices_A_df[prices_A_df['Supplier_Funder'] == 'Generics_GHSD']['Co_payment'].values[0]
    }
    prices_product_B = {
        'S1_G1': prices_B_df[prices_B_df['Supplier_Funder'] == 'Gilead_GF']['Price'].values[0],
        'S1_G2': prices_B_df[prices_B_df['Supplier_Funder'] == 'Gilead_GHSD']['Price'].values[0],
        'S2_G1': prices_B_df[prices_B_df['Supplier_Funder'] == 'Generics_GF']['Price'].values[0],
        'S2_G2': prices_B_df[prices_B_df['Supplier_Funder'] == 'Generics_GHSD']['Price'].values[0]
    }
    co_payment_product_B = {
        'S1_G1': prices_B_df[prices_B_df['Supplier_Funder'] == 'Gilead_GF']['Co_payment'].values[0],
        'S1_G2': prices_B_df[prices_B_df['Supplier_Funder'] == 'Gilead_GHSD']['Co_payment'].values[0],
        'S2_G1': prices_B_df[prices_B_df['Supplier_Funder'] == 'Generics_GF']['Co_payment'].values[0],
        'S2_G2': prices_B_df[prices_B_df['Supplier_Funder'] == 'Generics_GHSD']['Co_payment'].values[0]
    }

    # Load demand - ensure correct order using COUNTRY_NAMES
    demand_A_df = pd.read_excel(CONFIG_PATH, sheet_name='LEN_vial_Demand')
    demand_B_df = pd.read_excel(CONFIG_PATH, sheet_name='LEN_tablet_Demand')

    demand_data_A = {q: [] for q in quarters}
    demand_data_B = {q: [] for q in quarters}

    for country in COUNTRY_NAMES:
        row_A = demand_A_df[demand_A_df['Country'] == country]
        row_B = demand_B_df[demand_B_df['Country'] == country]
        for q in quarters:
            demand_data_A[q].append(int(row_A[q].values[0]) if not row_A.empty else 0)
            demand_data_B[q].append(int(row_B[q].values[0]) if not row_B.empty else 0)

    # Load capacities
    capacity_df = pd.read_excel(CONFIG_PATH, sheet_name='Supplier_Capacities')
    supplier_capacity_table_product_A = pd.DataFrame({
        'Supplier 1': capacity_df['Gilead_Capacity_LEN_vial'].values,
        'Supplier 2': capacity_df['Generics_Capacity_LEN_vial'].values
    }, index=capacity_df['Quarter'].values)
    supplier_capacity_table_product_B = pd.DataFrame({
        'Supplier 1': capacity_df['Gilead_Capacity_LEN_tablet'].values,
        'Supplier 2': capacity_df['Generics_Capacity_LEN_tablet'].values
    }, index=capacity_df['Quarter'].values)

    # Load share constraints
    share_df = pd.read_excel(CONFIG_PATH, sheet_name='Share_Constraints')
    supplier_share_constraints = {}

    for product_display in ['LEN_vial', 'LEN_tablet']:
        product_key = 'Product_A' if product_display == 'LEN_vial' else 'Product_B'
        supplier_share_constraints[product_key] = {}

        for funder in ['GF', 'GHSD']:
            for supplier in ['Gilead', 'Generics']:
                row = share_df[(share_df['Product'] == product_display) &
                              (share_df['Funder'] == funder) &
                              (share_df['Supplier'] == supplier)]
                if not row.empty:
                    supplier_key = 'Supplier_1' if supplier == 'Gilead' else 'Supplier_2'
                    funder_key = 'G1' if funder == 'GF' else 'G2'

                    # Initialize nested dictionaries if they don't exist
                    if funder_key not in supplier_share_constraints[product_key]:
                        supplier_share_constraints[product_key][funder_key] = {}

                    supplier_share_constraints[product_key][funder_key][supplier_key] = {
                        'min': row['Min_Share_%'].values[0] / 100,
                        'max': row['Max_Share_%'].values[0] / 100
                    }

    # Load minimum volumes
    min_volume_df = pd.read_excel(CONFIG_PATH, sheet_name='Minimum_Volumes')
    min_volume_requirements = {}
    for _, row in min_volume_df.iterrows():
        product_display = row['Product']
        product_key = 'Product_A' if product_display == 'LEN_vial' else 'Product_B'
        supplier = 'Supplier_1' if row['Supplier'] == 'Gilead' else 'Supplier_2'
        funder = 'G1' if row['Funder'] == 'GF' else 'G2'
        min_vol = row['Min_Annual_Volume']
        if min_vol > 0:
            min_volume_requirements.setdefault(product_key, {}).setdefault(supplier, {})[funder] = min_vol

    # Load country funder split constraints
    country_split_df = pd.read_excel(CONFIG_PATH, sheet_name='Country_Funder_Split')
    country_funder_constraints = {}
    for _, row in country_split_df.iterrows():
        product_display = row['Product']
        product_key = 'Product_A' if product_display == 'LEN_vial' else 'Product_B'
        country_name = row['Country']
        if country_name in COUNTRY_NAMES:
            country_idx = COUNTRY_NAMES.index(country_name)
            country_funder_constraints.setdefault(product_key, {})[country_idx] = {
                'min_gf': row['Min_GF_Share_%'] / 100,
                'max_gf': row['Max_GF_Share_%'] / 100,
                'min_ghsd': row['Min_GHSD_Share_%'] / 100,
                'max_ghsd': row['Max_GHSD_Share_%'] / 100
            }

    return (prices_product_A, co_payment_product_A, prices_product_B, co_payment_product_B,
            demand_data_A, demand_data_B,
            supplier_capacity_table_product_A, supplier_capacity_table_product_B,
            supplier_share_constraints, min_volume_requirements, country_funder_constraints)

# ========== FUNCTION TO SAVE DETAILED RESULTS ==========
def save_detailed_results_to_dashboard(writer, solution, prices, co_payment, demand_df,
                                       supplier_capacity_table, toggle_matrix,
                                       product_name, scenario_name):
    solution_prod = solution.reshape(n_quarters, n_countries, n_combinations)

    s1g1 = pd.DataFrame(solution_prod[:, :, 0], index=quarters, columns=countries_original)
    s1g2 = pd.DataFrame(solution_prod[:, :, 1], index=quarters, columns=countries_original)
    s2g1 = pd.DataFrame(solution_prod[:, :, 2], index=quarters, columns=countries_original)
    s2g2 = pd.DataFrame(solution_prod[:, :, 3], index=quarters, columns=countries_original)

    g1_units_s1 = s1g1.sum().sum()
    g1_units_s2 = s2g1.sum().sum()
    g2_units_s1 = s1g2.sum().sum()
    g2_units_s2 = s2g2.sum().sum()
    total_units = g1_units_s1 + g1_units_s2 + g2_units_s1 + g2_units_s2

    g1_cost = (g1_units_s1 * prices['S1_G1']) + (g1_units_s2 * prices['S2_G1'])
    g2_cost = (g2_units_s1 * prices['S1_G2']) + (g2_units_s2 * prices['S2_G2'])
    co_payment_amount = (g1_units_s1 * co_payment['S1_G1'])

    product_display = 'LEN_vial' if 'ProductA' in product_name else 'LEN_tablet'

    # Cost Build-Up
    cost_build_up = pd.DataFrame({
        'Component': ['G1 Cost', 'G2 Cost', 'Co-payment', 'TOTAL COST'],
        'Amount ($)': [g1_cost, g2_cost, co_payment_amount, g1_cost + g2_cost + co_payment_amount]
    })
    cost_build_up.to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_Cost', index=False)

    # Supplier Totals
    supplier_summary = pd.DataFrame({
        'Quarter': quarters,
        'Gilead GF': s1g1.sum(axis=1).values,
        'Gilead GHSD': s1g2.sum(axis=1).values,
        'Generics GF': s2g1.sum(axis=1).values,
        'Generics GHSD': s2g2.sum(axis=1).values
    })
    supplier_summary['Gilead Total'] = supplier_summary['Gilead GF'] + supplier_summary['Gilead GHSD']
    supplier_summary['Generics Total'] = supplier_summary['Generics GF'] + supplier_summary['Generics GHSD']
    supplier_summary.loc['TOTAL'] = supplier_summary.sum(numeric_only=True)
    supplier_summary.to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_SupplierTotals', index=False)

    # GF Supply by Country
    g1_supply = pd.DataFrame(solution_prod[:, :, 0] + solution_prod[:, :, 2], index=quarters, columns=countries_original)
    g1_supply.loc['Total'] = g1_supply.sum()
    translate_dataframe_countries(g1_supply).to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_GF_ByCountry')

    # GHSD Supply by Country
    g2_supply = pd.DataFrame(solution_prod[:, :, 1] + solution_prod[:, :, 3], index=quarters, columns=countries_original)
    g2_supply.loc['Total'] = g2_supply.sum()
    translate_dataframe_countries(g2_supply).to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_GHSD_ByCountry')

    # Country Split Analysis
    country_split_analysis = []
    for c_idx, country_code in enumerate(countries_original):
        country_name = translate_country_name(country_code)
        country_gf = (s1g1[country_code].sum() + s2g1[country_code].sum())
        country_ghsd = (s1g2[country_code].sum() + s2g2[country_code].sum())
        country_total = country_gf + country_ghsd
        if country_total > 0:
            gf_pct = (country_gf / country_total) * 100
            ghsd_pct = (country_ghsd / country_total) * 100
        else:
            gf_pct = ghsd_pct = 0
        country_split_analysis.append({
            'Country': country_name,
            'Total Units': f"{country_total:,.0f}",
            'GF Units': f"{country_gf:,.0f}",
            'GF %': round(gf_pct, 1),
            'GHSD Units': f"{country_ghsd:,.0f}",
            'GHSD %': round(ghsd_pct, 1)
        })
    pd.DataFrame(country_split_analysis).to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_CountrySplit', index=False)

    # Gilead Detailed
    s1_detailed = pd.DataFrame(index=quarters, columns=countries_original)
    for q in range(n_quarters):
        for c in range(n_countries):
            s1_detailed.iloc[q, c] = solution_prod[q, c, 0] + solution_prod[q, c, 1]
    s1_detailed.loc['Total'] = s1_detailed.sum()
    translate_dataframe_countries(s1_detailed).to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_Gilead_Total')

    # Generics Detailed
    s2_detailed = pd.DataFrame(index=quarters, columns=countries_original)
    for q in range(n_quarters):
        for c in range(n_countries):
            s2_detailed.iloc[q, c] = solution_prod[q, c, 2] + solution_prod[q, c, 3]
    s2_detailed.loc['Total'] = s2_detailed.sum()
    translate_dataframe_countries(s2_detailed).to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_Generics_Total')

    # Demand Fulfillment
    total_supplied = solution_prod.sum(axis=2).sum(axis=1)
    demand_check = pd.DataFrame({
        'Quarter': quarters,
        'Demand': demand_df.sum(axis=1).values,
        'Supplied': total_supplied,
        'Difference': demand_df.sum(axis=1).values - total_supplied
    })
    demand_check.to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_DemandCheck', index=False)

    # Capacity Utilization
    capacity_util = pd.DataFrame(index=quarters)
    for q, quarter in enumerate(quarters):
        s1_usage = s1g1.iloc[q].sum() + s1g2.iloc[q].sum()
        s2_usage = s2g1.iloc[q].sum() + s2g2.iloc[q].sum()
        s1_cap = supplier_capacity_table.loc[quarter, 'Supplier 1']
        s2_cap = supplier_capacity_table.loc[quarter, 'Supplier 2']
        capacity_util.loc[quarter, 'Gilead Usage'] = s1_usage
        capacity_util.loc[quarter, 'Gilead Capacity'] = s1_cap
        capacity_util.loc[quarter, 'Gilead Util %'] = (s1_usage / s1_cap * 100) if s1_cap > 0 else 0
        capacity_util.loc[quarter, 'Generics Usage'] = s2_usage
        capacity_util.loc[quarter, 'Generics Capacity'] = s2_cap
        capacity_util.loc[quarter, 'Generics Util %'] = (s2_usage / s2_cap * 100) if s2_cap > 0 else 0
    capacity_util.to_excel(writer, sheet_name=f'{product_display}_{scenario_name}_CapacityUtil')

    return {
        'total_units': total_units,
        'total_cost': g1_cost + g2_cost + co_payment_amount,
        'g1_units': g1_units_s1 + g1_units_s2,
        'g2_units': g2_units_s1 + g2_units_s2,
        's1_g1_units': g1_units_s1,
        's2_g1_units': g1_units_s2,
        's1_g2_units': g2_units_s1,
        's2_g2_units': g2_units_s2
    }

# ========== FUNCTION TO RUN OPTIMIZATION ==========
def run_multi_product_optimization(prices_list, total_cost_list, demand_dfs, supplier_capacity_tables,
                                   toggle_matrices, enable_smoothing=False, smoothing_params=None,
                                   enable_share_constraints=False, share_constraints=None,
                                   enable_min_volume=False, min_volume_requirements=None,
                                   enable_country_split=False, country_split_constraints=None):

    n_products = len(prices_list)
    n_vars_per_product = n_quarters * n_countries * n_combinations
    n_vars_total = n_products * n_vars_per_product
    n_vars_total_with_smoothing = n_vars_total

    c = []
    for product_idx in range(n_products):
        for q in range(n_quarters):
            for ctry in range(n_countries):
                c.extend([total_cost_list[product_idx]['S1_G1'],
                         total_cost_list[product_idx]['S1_G2'],
                         total_cost_list[product_idx]['S2_G1'],
                         total_cost_list[product_idx]['S2_G2']])

    constraints = []

    for product_idx in range(n_products):
        product_offset = product_idx * n_vars_per_product

        # Demand fulfillment
        for q in range(n_quarters):
            for ctry in range(n_countries):
                A_row = np.zeros(n_vars_total_with_smoothing)
                for comb in range(n_combinations):
                    idx = product_offset + q * n_countries * n_combinations + ctry * n_combinations + comb
                    A_row[idx] = 1
                constraints.append({'type': 'eq', 'A': A_row, 'b': demand_dfs[product_idx].iloc[q, ctry]})

        # Supplier capacity
        for q in range(n_quarters):
            quarter_name = quarters[q]
            A_row_s1 = np.zeros(n_vars_total_with_smoothing)
            A_row_s2 = np.zeros(n_vars_total_with_smoothing)
            for ctry in range(n_countries):
                idx_g1 = product_offset + q * n_countries * n_combinations + ctry * n_combinations + 0
                idx_g2 = product_offset + q * n_countries * n_combinations + ctry * n_combinations + 1
                idx_g3 = product_offset + q * n_countries * n_combinations + ctry * n_combinations + 2
                idx_g4 = product_offset + q * n_countries * n_combinations + ctry * n_combinations + 3
                A_row_s1[idx_g1] = 1
                A_row_s1[idx_g2] = 1
                A_row_s2[idx_g3] = 1
                A_row_s2[idx_g4] = 1
            constraints.append({'type': 'ineq', 'A': A_row_s1, 'b': supplier_capacity_tables[product_idx].loc[quarter_name, 'Supplier 1']})
            constraints.append({'type': 'ineq', 'A': A_row_s2, 'b': supplier_capacity_tables[product_idx].loc[quarter_name, 'Supplier 2']})

        # G1 mechanism capacity
        for q in range(n_quarters):
            A_row_g1 = np.zeros(n_vars_total_with_smoothing)
            for ctry in range(n_countries):
                idx_s1g1 = product_offset + q * n_countries * n_combinations + ctry * n_combinations + 0
                idx_s2g1 = product_offset + q * n_countries * n_combinations + ctry * n_combinations + 2
                A_row_g1[idx_s1g1] = 1
                A_row_g1[idx_s2g1] = 1
            total_g1_capacity = supplier_capacity_tables[product_idx].loc[quarters[q], 'Supplier 1'] + supplier_capacity_tables[product_idx].loc[quarters[q], 'Supplier 2']
            constraints.append({'type': 'ineq', 'A': A_row_g1, 'b': total_g1_capacity})

        # Toggle restrictions
        for q in range(n_quarters):
            quarter_name = quarters[q]
            for comb in range(n_combinations):
                if toggle_matrices[product_idx][quarter_name][comb] == 0:
                    for ctry in range(n_countries):
                        A_row = np.zeros(n_vars_total_with_smoothing)
                        idx = product_offset + q * n_countries * n_combinations + ctry * n_combinations + comb
                        A_row[idx] = 1
                        constraints.append({'type': 'eq', 'A': A_row, 'b': 0})

    # Supplier share constraints
    if enable_share_constraints and share_constraints:
        for product_idx in range(n_products):
            product_offset = product_idx * n_vars_per_product
            product_name = 'Product_A' if product_idx == 0 else 'Product_B'
            if product_name in share_constraints:
                for funder in ['G1', 'G2']:
                    funder_display = 'GF' if funder == 'G1' else 'GHSD'
                    if funder_display in share_constraints[product_name]:
                        comb_indices = [0, 2] if funder == 'G1' else [1, 3]
                        for supplier_name, shares in share_constraints[product_name][funder_display].items():
                            supplier_idx = 0 if supplier_name == 'Supplier_1' else 1
                            comb_idx = 0 if (supplier_idx == 0 and funder == 'G1') else (1 if (supplier_idx == 0 and funder == 'G2') else (2 if (supplier_idx == 1 and funder == 'G1') else 3))
                            A_supplier = np.zeros(n_vars_total_with_smoothing)
                            A_total = np.zeros(n_vars_total_with_smoothing)
                            for q in range(n_quarters):
                                for ctry in range(n_countries):
                                    idx_supplier = product_offset + q * n_countries * n_combinations + ctry * n_combinations + comb_idx
                                    A_supplier[idx_supplier] = 1
                                    for comb in comb_indices:
                                        idx_total = product_offset + q * n_countries * n_combinations + ctry * n_combinations + comb
                                        A_total[idx_total] = 1
                            if shares['min'] > 0:
                                constraints.append({'type': 'ineq', 'A': A_supplier - shares['min'] * A_total, 'b': 0})
                            if shares['max'] < 1:
                                constraints.append({'type': 'ineq', 'A': -A_supplier + shares['max'] * A_total, 'b': 0})

    # Minimum volume requirements
    if enable_min_volume and min_volume_requirements:
        for product_name, suppliers in min_volume_requirements.items():
            product_idx = 0 if product_name == 'Product_A' else 1
            product_offset = product_idx * n_vars_per_product
            for supplier_name, funders in suppliers.items():
                supplier_idx = 0 if supplier_name == 'Supplier_1' else 1
                for funder, min_volume in funders.items():
                    comb_idx = 0 if (supplier_idx == 0 and funder == 'G1') else (1 if (supplier_idx == 0 and funder == 'G2') else (2 if (supplier_idx == 1 and funder == 'G1') else 3))
                    A_min_vol = np.zeros(n_vars_total_with_smoothing)
                    for q in range(n_quarters):
                        for ctry in range(n_countries):
                            idx = product_offset + q * n_countries * n_combinations + ctry * n_combinations + comb_idx
                            A_min_vol[idx] = 1
                    constraints.append({'type': 'ineq', 'A': -A_min_vol, 'b': -min_volume})

    # Country funder split constraints (FIXED VERSION)
    if enable_country_split and country_split_constraints:
        for product_idx in range(n_products):
            product_offset = product_idx * n_vars_per_product
            product_name = 'Product_A' if product_idx == 0 else 'Product_B'
            if product_name in country_split_constraints:
                for country_idx, constraints_dict in country_split_constraints[product_name].items():
                    A_gf_total = np.zeros(n_vars_total_with_smoothing)
                    A_ghsd_total = np.zeros(n_vars_total_with_smoothing)
                    A_total = np.zeros(n_vars_total_with_smoothing)
                    for q in range(n_quarters):
                        for comb in [0, 2]:
                            idx = product_offset + q * n_countries * n_combinations + country_idx * n_combinations + comb
                            A_gf_total[idx] = 1
                            A_total[idx] = 1
                        for comb in [1, 3]:
                            idx = product_offset + q * n_countries * n_combinations + country_idx * n_combinations + comb
                            A_ghsd_total[idx] = 1
                            A_total[idx] = 1

                    # GF constraints
                    if constraints_dict['min_gf'] > 0:
                        constraints.append({'type': 'ineq', 'A': A_gf_total - constraints_dict['min_gf'] * A_total, 'b': 0})
                    if constraints_dict['max_gf'] < 1:
                        if constraints_dict['max_gf'] == 0:
                            constraints.append({'type': 'eq', 'A': A_gf_total, 'b': 0})
                        else:
                            constraints.append({'type': 'ineq', 'A': A_gf_total - constraints_dict['max_gf'] * A_total, 'b': 0})

                    # GHSD constraints
                    if constraints_dict['min_ghsd'] > 0:
                        constraints.append({'type': 'ineq', 'A': A_ghsd_total - constraints_dict['min_ghsd'] * A_total, 'b': 0})
                    if constraints_dict['max_ghsd'] < 1:
                        if constraints_dict['max_ghsd'] == 0:
                            constraints.append({'type': 'eq', 'A': A_ghsd_total, 'b': 0})
                        else:
                            constraints.append({'type': 'ineq', 'A': A_ghsd_total - constraints_dict['max_ghsd'] * A_total, 'b': 0})

    # Build matrices
    A_ub, b_ub, A_eq, b_eq = [], [], [], []
    for const in constraints:
        if const['type'] == 'ineq':
            A_ub.append(const['A'])
            b_ub.append(const['b'])
        else:
            A_eq.append(const['A'])
            b_eq.append(const['b'])

    bounds = [(0, None) for _ in range(n_vars_total_with_smoothing)]
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    return result, n_vars_per_product

# ========== MAIN EXECUTION ==========
print("\n" + "="*80)
print("OPTIMIZATION WITH SEPARATE PRODUCT DEMAND")
print(f"Products: LEN_vial and LEN_tablet")
print(f"Countries: {', '.join(COUNTRY_NAMES[:5])}... + {len(COUNTRY_NAMES)-12} placeholders")
print("="*80)

# Check if config file exists
if not os.path.exists(CONFIG_PATH):
    print("\n" + "="*80)
    print("CONFIGURATION FILE NOT FOUND")
    print("="*80)
    create_configuration_excel()
    print("\n" + "="*80)
    print("NEXT STEPS:")
    print("1. Open the file: optimization_config.xlsx")
    print("2. Review and edit values as needed")
    print("3. Save the file")
    print("4. Re-run this script")
    print("="*80)
    exit()

# Load configuration
(prices_product_A, co_payment_product_A, prices_product_B, co_payment_product_B,
 demand_data_A, demand_data_B,
 supplier_capacity_table_product_A, supplier_capacity_table_product_B,
 supplier_share_constraints, min_volume_requirements, country_funder_constraints) = load_configuration()

print("\n✓ Configuration loaded successfully")
print(f"  LEN_vial - Gilead GF Price: ${prices_product_A['S1_G1']:.2f} + Co-payment: ${co_payment_product_A['S1_G1']:.2f}")
print(f"  LEN_tablet - Gilead GF Price: ${prices_product_B['S1_G1']:.2f} + Co-payment: ${co_payment_product_B['S1_G1']:.2f}")

# Calculate total demand
total_demand_A = sum(sum(demand_data_A[q]) for q in quarters)
total_demand_B = sum(sum(demand_data_B[q]) for q in quarters)
print(f"\nTotal Annual Demand:")
print(f"  LEN_vial: {total_demand_A:,.0f} units")
print(f"  LEN_tablet: {total_demand_B:,.0f} units")

# Prepare baseline data
demand_df_A_base = pd.DataFrame(apply_demand_scenario(demand_data_A, 'base'), index=countries_original).T
demand_df_B_base = pd.DataFrame(apply_demand_scenario(demand_data_B, 'base'), index=countries_original).T

# Ensure columns are in correct order
demand_df_A_base = demand_df_A_base[countries_original]
demand_df_B_base = demand_df_B_base[countries_original]

# Toggle matrices
toggle_matrix_product_A = {quarter: [1, 1, 1, 1] for quarter in quarters}
toggle_matrix_product_A['Q1'][3] = 0
toggle_matrix_product_A['Q2'][3] = 0
toggle_matrix_product_A['Q3'][3] = 0
toggle_matrix_product_B = toggle_matrix_product_A.copy()

# Run optimization
total_cost_A = {k: prices_product_A[k] + co_payment_product_A[k] for k in prices_product_A}
total_cost_B = {k: prices_product_B[k] + co_payment_product_B[k] for k in prices_product_B}

result_baseline, n_vars_per = run_multi_product_optimization(
    [prices_product_A, prices_product_B],
    [total_cost_A, total_cost_B],
    [demand_df_A_base, demand_df_B_base],
    [supplier_capacity_table_product_A, supplier_capacity_table_product_B],
    [toggle_matrix_product_A, toggle_matrix_product_B],
    enable_smoothing=ENABLE_PRODUCTION_SMOOTHING,
    smoothing_params=None,
    enable_share_constraints=ENABLE_SUPPLIER_SHARE_CONSTRAINTS,
    share_constraints=supplier_share_constraints,
    enable_min_volume=ENABLE_MINIMUM_VOLUME_CONSTRAINTS,
    min_volume_requirements=min_volume_requirements,
    enable_country_split=ENABLE_COUNTRY_FUNDER_SPLIT_CONSTRAINTS,
    country_split_constraints=country_funder_constraints
)

if result_baseline.success:
    print(f"\n✓ OPTIMIZATION SUCCESSFUL!")
    print(f"Total cost for both products: ${result_baseline.fun:,.2f}")

    solution_A = result_baseline.x[:n_vars_per]
    solution_B = result_baseline.x[n_vars_per:2*n_vars_per]

    # Save output files
    with pd.ExcelWriter(os.path.join(SAVE_PATH, 'LEN_vial_detailed_results.xlsx'), engine='openpyxl') as writer:
        save_detailed_results_to_dashboard(writer, solution_A, prices_product_A, co_payment_product_A,
                                          demand_df_A_base, supplier_capacity_table_product_A,
                                          toggle_matrix_product_A, 'ProductA', 'Full')

    with pd.ExcelWriter(os.path.join(SAVE_PATH, 'LEN_tablet_detailed_results.xlsx'), engine='openpyxl') as writer:
        save_detailed_results_to_dashboard(writer, solution_B, prices_product_B, co_payment_product_B,
                                          demand_df_B_base, supplier_capacity_table_product_B,
                                          toggle_matrix_product_B, 'ProductB', 'Full')

    print("\n✓ Output files saved:")
    print("  • LEN_vial_detailed_results.xlsx")
    print("  • LEN_tablet_detailed_results.xlsx")

else:
    print(f"\n✗ OPTIMIZATION FAILED: {result_baseline.message}")

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE")
print("="*80)


Files will be saved to: /content

OPTIMIZATION WITH SEPARATE PRODUCT DEMAND
Products: LEN_vial and LEN_tablet
Countries: Eswatini, Kenya, Lesotho, Malawi, Mozambique... + 4 placeholders

✓ Configuration loaded successfully
  LEN_vial - Gilead GF Price: $21.05 + Co-payment: $23.86
  LEN_tablet - Gilead GF Price: $8.95 + Co-payment: $10.14

Total Annual Demand:
  LEN_vial: 8,000,002 units
  LEN_tablet: 5,713,425 units

✓ OPTIMIZATION SUCCESSFUL!
Total cost for both products: $150,080,926.18

✓ Output files saved:
  • LEN_vial_detailed_results.xlsx
  • LEN_tablet_detailed_results.xlsx

OPTIMIZATION COMPLETE
